## Lab 5: AgentCore Evaluations - Online Evaluation for Customer Support Agent

### Overview

This lab demonstrates how to use AgentCore Evaluations to continuously monitor your production customer support agent from Lab 4. You'll configure online evaluation to automatically assess agent performance in real-time as customers interact with it.

**Workshop Journey:**

- **Lab 1 (Done):** Create Agent Prototype - Built a functional customer support agent
- **Lab 2 (Done):** Enhance with Memory - Added conversation context and personalization
- **Lab 3 (Done):** Scale with Gateway & Identity - Shared tools across agents securely
- **Lab 4 (Done):** Deploy to Production - Used AgentCore Runtime with observability
- **Lab 5 (Current):** Evaluate Agent Performance - Monitor quality with online evaluations
- **Lab 6:** Build User Interface - Create a customer-facing application

### What You'll Learn

You'll configure online evaluation with built-in evaluators, generate test interactions, and analyze quality metrics through AgentCore Observability dashboards to improve agent performance.

### Online Evaluation Overview

Online evaluation continuously monitors deployed agents in production, unlike on-demand evaluation which analyzes specific selected interactions. It consists of three components: session sampling with configurable rules, multiple evaluation methods (built-in or custom evaluators), and monitoring through dashboards with quality trends and low-scoring session investigation.

Since your agent runs on AgentCore Runtime, AgentCore Observability automatically instruments the code and provides comprehensive logs and traces using [OTEL](https://opentelemetry.io/) instrumentation.

### Prerequisites

Complete Lab 4 to have the customer support agent deployed. You'll need AWS account access to Amazon Bedrock AgentCore with Evaluations permissions.

### Architecture
<div style="text-align:left">
    <img src="images/architecture_lab5_evaluation.png" width="75%"/>
</div>

*Online evaluation automatically monitors agent interactions, applies evaluators based on sampling rules, and outputs results to CloudWatch for analysis.*

### Step 1: Import Required Libraries and Initialize Clients

In [1]:
!pip install -r requirements.txt

In [23]:
from bedrock_agentcore_starter_toolkit import Evaluation, Runtime
import json
import uuid
from pathlib import Path
from boto3.session import Session
from IPython.display import Markdown, display
from lab_helpers.utils import get_ssm_parameter, get_or_create_cognito_pool

In [24]:
import os

# Setup boto session with profile
boto_session = Session(profile_name='test')
region = boto_session.region_name or 'us-east-1'

# Extract and set credentials as environment variables
credentials = boto_session.get_credentials()
os.environ['AWS_ACCESS_KEY_ID'] = credentials.access_key
os.environ['AWS_SECRET_ACCESS_KEY'] = credentials.secret_key
if credentials.token:
    os.environ['AWS_SESSION_TOKEN'] = credentials.token
os.environ['AWS_DEFAULT_REGION'] = region

# Remove AWS_PROFILE to ensure explicit credentials are used
if 'AWS_PROFILE' in os.environ:
    del os.environ['AWS_PROFILE']

print(f"Using AWS region: {region}")
print(f"Using profile: test")

Using AWS region: us-east-1
Using profile: test


In [25]:
eval_client = Evaluation(region=region)
runtime_client = Runtime()

### Step 2: Retrieve Agent Information from Lab 4

Retrieve the customer support agent ARN from SSM Parameter Store where it was saved during Lab 4 deployment.

In [26]:
try:
    # Get agent ARN from SSM parameter store (saved in Lab 4)
    agent_arn = get_ssm_parameter("/app/customersupport/agentcore/runtime_arn")
    
    # Extract agent ID from ARN
    agent_id = agent_arn.split(":")[-1].split("/")[-1]
    
    # Set runtime client config path
    runtime_client._config_path = Path.cwd() / ".bedrock_agentcore.yaml"
    
    print("Agent ID:", agent_id)
    print("Agent ARN:", agent_arn)
except Exception as e:
    raise Exception(f"""Missing agent information from Lab 4. Please run lab-04-agentcore-runtime.ipynb first. Error: {str(e)}""")

Agent ID: customer_support_agent-DG2QVM6bzx
Agent ARN: arn:aws:bedrock-agentcore:us-east-1:145492107390:runtime/customer_support_agent-DG2QVM6bzx


### Step 3: Create Online Evaluation Configuration

Now let's create an online evaluation configuration for our customer support agent. We'll use built-in evaluators to assess different aspects of agent performance:

- **Builtin.GoalSuccessRate** - Measures how well the agent achieves user goals
- **Builtin.Correctness** - Evaluates factual accuracy of responses
- **Builtin.ToolSelectionAccuracy** - Evaluates appropriate tool selection

We'll set the sampling rate to 100% for demonstration purposes, but in production you might use a lower rate (e.g., 10-20%) based on your traffic volume.

In [37]:
# Create or reuse online evaluation configuration
try:
    # Try to create new configuration
    response = eval_client.create_online_config(
        agent_id=agent_id,
        config_name="customer_support_agent_eval",
        sampling_rate=100,  # Evaluate 100% of sessions for demo
        evaluator_list=[
            "Builtin.GoalSuccessRate", 
            "Builtin.Correctness",
            "Builtin.ToolSelectionAccuracy"
        ],
        config_description="Customer support agent online evaluation",
        auto_create_execution_role=True
    )
    print("Online evaluation configuration created successfully!")
    print(f"Configuration ID: {response['onlineEvaluationConfigId']}")
    
except Exception as e:
    # If config already exists, find and use it
    if "ConflictException" in str(e) or "already exists" in str(e):
        print("Configuration already exists, retrieving existing config...")
        
        # List all configs and find ours by name
        all_configs = eval_client.list_online_configs()
        existing_config = None
        
        for config in all_configs.get('onlineEvaluationConfigs', []):
            # Match by config name (list_online_configs doesn't return agentId)
            if config.get('onlineEvaluationConfigName') == "customer_support_agent_eval":
                existing_config = config
                break
        
        if existing_config:
            config_id = existing_config['onlineEvaluationConfigId']
            config_name = existing_config['onlineEvaluationConfigName']
            config_status = existing_config.get('status', 'UNKNOWN')
            print(f"✓ Using existing configuration: {config_name}")
            print(f"  Configuration ID: {config_id}")
            print(f"  Status: {config_status}")
            response = {'onlineEvaluationConfigId': config_id}
        else:
            print(f"Could not find config 'customer_support_agent_eval'. Available configs:")
            for cfg in all_configs.get('onlineEvaluationConfigs', []):
                print(f"  - {cfg.get('onlineEvaluationConfigName')}")
            raise Exception("Configuration exists but could not be retrieved")
    else:
        # Different error, re-raise
        raise

Creating online evaluation config: customer_support_agent_eval for agent: customer_support_agent-DG2QVM6bzx
Configuration: sampling_rate=100.0%, evaluators=['Builtin.GoalSuccessRate', 'Builtin.Correctness', 'Builtin.ToolSelectionAccuracy']
Creating online evaluation config: customer_support_agent_eval for agent: customer_support_agent-DG2QVM6bzx
Auto-creating execution role for config: customer_support_agent_eval
Getting or creating evaluation execution role for config: customer_support_agent_eval
Using AWS region: us-east-1, account ID: 145492107390
Role name: AgentCoreEvalsSDK-us-east-1-d04ba7b68b
✅ Reusing existing evaluation execution role: arn:aws:iam::145492107390:role/AgentCoreEvalsSDK-us-east-1-d04ba7b68b
✓ Execution role ready: arn:aws:iam::145492107390:role/AgentCoreEvalsSDK-us-east-1-d04ba7b68b
Listing all online evaluation configs
Found 1 online evaluation config(s)


Configuration already exists, retrieving existing config...


✓ Using existing configuration: customer_support_agent_eval
  Configuration ID: customer_support_agent_eval-JzK6oV86nL
  Status: ACTIVE


### Step 4: Verify Configuration Status

Verify the evaluation configuration is properly created and enabled by retrieving its details.

In [38]:
config_details = eval_client.get_online_config(config_id=response['onlineEvaluationConfigId'])
print("Configuration Details:")
print(json.dumps(config_details, indent=2, default=str))

Configuration Details:
{
  "ResponseMetadata": {
    "RequestId": "be6de67b-6aff-432c-be48-6e12260bc8a0",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 29 Dec 2025 17:22:37 GMT",
      "content-type": "application/json",
      "content-length": "1049",
      "connection": "keep-alive",
      "x-amzn-requestid": "be6de67b-6aff-432c-be48-6e12260bc8a0",
      "x-amzn-remapped-x-amzn-requestid": "82b10dcc-e258-4cc6-b14c-8f9de3f3fd5c",
      "x-amzn-remapped-content-length": "1049",
      "x-amzn-remapped-connection": "keep-alive",
      "x-cache": "Miss from cloudfront",
      "via": "1.1 f588325f7617672d954c4267c8bee1ea.cloudfront.net (CloudFront)",
      "x-amz-cf-id": "dQKgWqbhXiA33IgWMrjGNnk00q3TVryyKlXsQJhnvYbtu1HhZbNYOw==",
      "x-amz-apigw-id": "WXHSmHcUIAMEuhA=",
      "x-amzn-trace-id": "Root=1-6952b8dd-5df87b5c1b0f51fa72eed591",
      "x-amz-cf-pop": "IAD12-P1",
      "x-amzn-remapped-date": "Mon, 29 Dec 2025 17:22:37 GMT"
    },
    "RetryAttempts": 0
  }

### Step 5: Generate Test Interactions

Invoke the customer support agent with various queries to generate traces for evaluation. Different test scenarios will demonstrate how the evaluators assess agent performance.

In [39]:
# Get authentication token
access_token = get_or_create_cognito_pool(refresh_token=True)
print(f"Access token obtained: {access_token['bearer_token'][:20]}...")

def invoke_agent_runtime(prompt, session_id=None):
    """Invoke the agent runtime using starter toolkit"""
    if not session_id:
        session_id = str(uuid.uuid4())
    
    response = runtime_client.invoke(
        payload={"prompt": prompt},
        session_id=session_id,
        bearer_token=access_token['bearer_token']
    )
    
    return response, session_id

Access token obtained: eyJraWQiOiI1K0VNVHNC...


#### Test Scenario 1: Product Information Query

In [40]:
session1 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I need information about the Gaming Console Pro. What are its specifications and price?",
    session1
)
print("Customer Query: Product information request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Product information request


"Based on the search results, here's the information about the **Gaming Console Pro** (PlayStation 5 Pro):

## **Key Specifications:**
- **Storage:** 2TB or 3TB Ultra Fast SSD options
- **Graphics:** Advanced ray tracing with super sharp image clarity
- **Performance:** High frame rate gameplay
- **Format:** All-digital console (no disc drive)
- **Design:** Styled like the PS5 \"Slim\" model
- **Includes:** Controller and Controller Charger

## **Price:**
- **$699.99 USD** (or $700)

## **Release Date:**
- November 7, 2024

## **Key Features:**
- Upgraded processing power compared to standard PS5
- Enhanced visual quality with advanced ray tracing
- Faster load times with the ultra-fast SSD
- Available in 2TB and 3TB storage options

I also notice from your context that you own a Gaming Console Pro (serial number MNO33333333) purchased on November 25, 2023. Since your device was purchased before the PS5 Pro was officially released in November 2024, I'd like to help you with any specific questions about your console.

**Can I assist you with:**
- Warranty information for your device?
- Technical support or connectivity issues?
- Return or exchange policies?
- Any other specifications about your console?"

#### Test Scenario 2: Technical Support Request

In [12]:
session2 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "My laptop won't start up. Can you help me troubleshoot this issue?",
    session2
)
print("Customer Query: Technical support request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Technical support request


"Great! I found some troubleshooting guidance. Here are the recommended steps for when a laptop won't start up:

## Basic Troubleshooting Steps:

1. **Check Power Connections**
   - Ensure the power cable is firmly connected to both the laptop and the power outlet
   - Verify the power outlet is working (try plugging in another device)
   - Look for any visible damage to the power cable

2. **Try a Different Power Cable**
   - If available, test with an alternative power cable to rule out cable issues

3. **Inspect for Physical Damage**
   - Check the laptop for any signs of physical damage or liquid spills
   - Make sure the charging port isn't damaged

## Additional Things to Check:

- **LED indicators**: Look for any lights on the keyboard or near the power button - they may indicate what's happening
- **Listen for sounds**: Note if you hear any fan noise or beeping codes (different beep patterns indicate different issues)
- **Battery level**: If it's been stored for a while, the battery may be completely drained - try leaving it plugged in for 30+ minutes before attempting to power on

## Next Steps:

If the above steps don't resolve the issue, it could indicate:
- A hardware problem with the motherboard or power supply
- A battery issue
- Internal component failure

**What I need to help you further:**
- Is there any power indicator light at all when you plug in the power cable?
- Do you hear any sounds (fan noise, beeping)?
- Does the laptop show any signs of power (keyboard lights, screen brightness, etc.)?

Also, if you have the laptop's serial number and would like to check your warranty status, I can help with that as well. This information would be useful if the device needs professional repair or service.

Is there anything specific about the startup issue you'd like me to help with?"

#### Test Scenario 3: Return Policy Inquiry

In [13]:
session3 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I bought a smartphone last week but it's not working properly. What's your return policy?",
    session3
)
print("Customer Query: Return policy inquiry")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Return policy inquiry


"Great news! Here's our return policy for smartphones:

**Return Policy Summary:**
- **Return Window:** 30 days from delivery
- **Condition Requirements:** Must be in original packaging with no physical damage, and requires factory reset
- **Process:** You can initiate a return through our online RMA portal or contact technical support
- **Refund Timeline:** 5-7 business days after inspection
- **Shipping:** We provide free return shipping with a prepaid label
- **Warranty:** Your smartphone includes a 1-year manufacturer warranty

**Since you purchased it last week, you're well within the 30-day return window!**

Before we proceed with a return, would you like me to help troubleshoot the issues you're experiencing? Sometimes we can resolve problems quickly. Could you tell me more about what's not working properly with your smartphone? That way, we might be able to get it working for you right away.

If troubleshooting doesn't help, I can guide you through the return process. Is there anything specific you'd like to know about the return policy or would you like technical support?"

#### Test Scenario 4: Complex Multi-Tool Query

In [14]:
session4 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I need help with my Gaming Console Pro. First, can you tell me about its warranty? Then I need technical support for connection issues.",
    session4
)
print("Customer Query: Complex multi-tool request")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Complex multi-tool request


"Great! I have the information you need. Here's what I found:

## **Warranty Status** 🛡️

Your Gaming Console Pro (Serial: MNO33333333) has an **expired warranty**:
- **Purchase Date:** November 25, 2023
- **Warranty End Date:** November 25, 2024
- **Status:** Expired (400 days ago)
- **Warranty Type:** Gaming Warranty
- **Original Coverage:** Controller issues, overheating protection, and hard drive replacement

Unfortunately, your warranty has expired, but extended warranty options may be available. Contact our support team for repair service pricing.

---

## **Connection Issues - Troubleshooting Guide** 🔧

Based on your connection problems, here are the recommended solutions:

### **Wi-Fi Connection Problems:**
1. **Restart your router and Gaming Console Pro**
2. **Check Wi-Fi password accuracy** - Verify you're entering the correct password
3. **Update network drivers** - Check for system updates
4. **Reset network settings** - Sometimes a fresh connection helps

### **Bluetooth Pairing Issues** (if applicable):
1. **Clear Bluetooth cache** - Remove paired devices and start fresh
2. **Remove and re-pair devices** - Go through the pairing process again
3. **Check compatibility** - Ensure your accessories are compatible
4. **Update Bluetooth drivers** - Install the latest firmware updates

### **General Tips:**
- Move closer to your router to check for signal strength issues
- Check for interference from other wireless devices
- Update your Gaming Console Pro firmware to the latest version

---

**Would you like me to provide more detailed steps for any of these troubleshooting methods? Or if you'd like to explore repair options or extended warranty coverage despite the expired warranty, I can connect you with our support team.**"

#### Test Scenario 5: General Capability Query

In [15]:
session5 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "What kind of support can you provide? List all your available tools and capabilities.",
    session5
)
print("Customer Query: Capability inquiry")
display(Markdown(response["response"].replace('\\n', '\n')))

Using JWT authentication


Customer Query: Capability inquiry


"

# Available Support & Tools

I'm here to help with a comprehensive range of customer support needs! Here are all the tools and capabilities I have available:

## **1. Product Information** 📋
- **Tool:** `get_product_info()`
- **What I can do:** Retrieve detailed technical specifications, features, warranty details, and policies for electronics products
- **Product types:** Laptops, smartphones, headphones, monitors, gaming consoles, and more
- **Useful for:** Comparing specs, understanding features, checking warranty coverage

## **2. Return & Warranty Policies** 🔄
- **Tool:** `get_return_policy()`
- **What I can do:** Provide comprehensive return policy details including timeframes, conditions, and procedures
- **Coverage:** Various product categories (smartphones, laptops, accessories, etc.)
- **Useful for:** Understanding return windows, refund processes, warranty terms

## **3. Warranty Status Verification** ✅
- **Tool:** `LambdaUsingSDK___check_warranty_status()`
- **What I can do:** Check if your product is still under warranty using the serial number
- **Optional:** Verify with your email address
- **Useful for:** Confirming warranty coverage for your specific device

## **4. Technical Support** 🔧
- **Tool:** `get_technical_support()`
- **What I can do:** Help troubleshoot technical issues and provide solutions
- **Useful for:** Device problems, configuration help, troubleshooting guides

## **5. Web Search for Updated Information** 🌐
- **Tool:** `LambdaUsingSDK___web_search()`
- **What I can do:** Search for current technical documentation, driver updates, specifications, and latest information
- **Useful for:** Finding recent updates, detailed documentation, comparative reviews

---

## **How I Can Help You:**

✅ Answer questions about product specifications and features  
✅ Check your warranty status (with your serial number)  
✅ Explain return and exchange policies  
✅ Troubleshoot technical issues  
✅ Find technical documentation and guides  
✅ Compare product capabilities  
✅ Direct you to appropriate resources if needed  

---

Since I see you're interested in **gaming headphones with low latency** for competitive FPS games, I'd be happy to help you find the right product or answer any technical questions about audio specifications like latency, frequency response, or connectivity options!

**What would you like help with today?** 🎮🎧"

### Step 6: Monitor Evaluation Results

Monitor evaluation results through the AgentCore Observability console. Results may take a few minutes to appear as the system processes traces and applies evaluators.

#### Accessing the Dashboard

1. Navigate to the [AgentCore Observability console](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents)
2. Find your customer support agent in the agents list
3. Click on the `DEFAULT` endpoint to view evaluation metrics
4. Look for the evaluation scores in the traces and sessions views

#### What You'll See

The dashboard will show:
- **Goal Success Rate**: How well the agent achieves customer objectives
- **Correctness**: Accuracy of information provided
- **Tool Selection Accuracy**: Appropriate tool choices for queries

![Online Evaluation Dashboard](images/online_evaluations_dashboard.png)

*Evaluation metrics displayed in the AgentCore Observability dashboard*

### Step 7: Understanding Evaluation Metrics

**Goal Success Rate** measures whether the agent successfully addresses the customer's primary intent. High scores indicate effective problem-solving; low scores suggest unmet needs, incomplete responses, or misunderstood requests.

**Correctness** evaluates factual accuracy of responses. High scores indicate accurate and reliable information; low scores suggest incorrect facts, outdated information, or misleading guidance.

**Tool Selection Accuracy** evaluates whether the agent chooses appropriate tools for each task. High scores indicate proper tool selection; low scores suggest wrong tools, unnecessary calls, or missing tool usage.

### Step 8: Analyzing Results and Next Steps

**For Low Goal Success Rates:** Refine the agent's system prompt, improve tool descriptions and parameters, and add specific training examples.

**For Low Correctness Scores:** Update the knowledge base with current information, improve fact-checking mechanisms, and review tool responses.

**For Tool-Related Issues:** Refine tool parameter schemas, improve tool selection logic, and enhance tool documentation.

**Continuous Monitoring:** Set up CloudWatch alarms for evaluation metrics, create dashboards for trend analysis, and implement automated alerts for quality degradation.

### Step 9: Clean Up (Optional)

Disable the online evaluation configuration if needed by uncommenting the code below.

In [ ]:
# Uncomment the following lines if you want to disable the evaluation configuration
# eval_client.delete_online_config(config_id=response['onlineEvaluationConfigId'])
# print("Online evaluation configuration disabled")

### Congratulations! 🎉

You have successfully completed **Lab 5: AgentCore Evaluations - Online Evaluation!**

### What You Accomplished

You configured automatic continuous online evaluation for your customer support agent with built-in evaluators assessing Goal Success Rate (customer satisfaction and problem resolution), Correctness (factual accuracy), and Tool Selection Accuracy (proper tool usage). Evaluation results are integrated with AgentCore Observability dashboards for real-time insights.

**Key Benefits:** Proactive quality assurance catches issues before customer impact, data-driven optimization guides improvements, production confidence through performance monitoring at scale, and continuous learning identifies patterns and opportunities.

**Next Steps:** Monitor your evaluation dashboard regularly, set up CloudWatch alarms for quality thresholds, use insights to iteratively improve your agent, and consider adding custom evaluators for domain-specific metrics.

### Next Up: [Lab 6: Build User Interface →](lab-06-frontend.ipynb)

Complete the customer experience by building a user-friendly web interface for customers to interact with your quality-monitored agent.

Your customer support agent is now production-ready with comprehensive quality monitoring! 🚀